In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_pinball_loss
import os
import json
import pandas as pd
import numpy as np
import xgboost as xgb
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import uniform, randint
from sklearn.metrics import mean_squared_error
import cloudpickle
import io

import warnings
warnings.filterwarnings("ignore")

# Net Revenue post-processing trainer for M2 (expected/bad/good quantiles)
# Target: revenue-per-1000-spend; Forecast outputs include both LTV/CAC and Net Revenue
# Models are saved in the current working directory with suffix _rev (same location as LTV models)


def calculate_current_month(row):
    cohort_start = row['cohort']
    elapsed_months = row['elapsed_months']
    return (cohort_start + pd.DateOffset(months=int(elapsed_months))).month


def quantile_loss(alpha):
    def custom_loss(y_true, y_pred):
        errors = y_pred - y_true
        grad = np.where(errors < 0, -alpha, 1 - alpha)
        hess = np.ones_like(y_true)
        return grad, hess
    return custom_loss


def add_arppu_features(df: pd.DataFrame, experiment_type: str) -> pd.DataFrame:
    """Add ARPPU-derived features to a copy of df, mirroring app_arppu.py behavior.
    - last_arppu: last available arppu for the cohort prior to current elapsed_months
    - arppu_change: relative change between last two months' arppu
    If 'arppu' is missing, features default to 0.
    """
    df_exp = df.copy()
    if 'arppu' not in df_exp.columns:
        # Ensure columns exist with zeros to keep pipeline consistent
        if experiment_type in ["arppu", "arppu_change"]:
            df_exp['last_arppu'] = 0.0
        if experiment_type == "arppu_change":
            df_exp['arppu_change'] = 0.0
        return df_exp

    if experiment_type in ["arppu", "arppu_change"]:
        df_exp = df_exp.sort_values(['cohort', 'elapsed_months']).reset_index(drop=True)
        df_exp['last_arppu'] = 0.0
        # For each cohort, set last_arppu as prior month's arppu
        for cohort_val in df_exp['cohort'].unique():
            cohort_slice = df_exp[df_exp['cohort'] == cohort_val].copy()
            prev_arppu = 0.0
            for idx in cohort_slice.index:
                em = df_exp.loc[idx, 'elapsed_months']
                if em > 0:
                    df_exp.loc[idx, 'last_arppu'] = float(prev_arppu)
                # update prev_arppu for next iteration
                val = df_exp.loc[idx, 'arppu'] if not pd.isna(df_exp.loc[idx, 'arppu']) else 0.0
                prev_arppu = val

    if experiment_type == "arppu_change":
        df_exp = df_exp.sort_values(['cohort', 'elapsed_months']).reset_index(drop=True)
        df_exp['arppu_change'] = 0.0
        for cohort_val in df_exp['cohort'].unique():
            cohort_slice = df_exp[df_exp['cohort'] == cohort_val]
            # compute change only when at least two historical months exist
            for idx in cohort_slice.index:
                em = int(df_exp.loc[idx, 'elapsed_months'])
                if em > 1:
                    prev_rows = cohort_slice[cohort_slice['elapsed_months'] < em]
                    if len(prev_rows) >= 2:
                        last_arppu = prev_rows.iloc[-1]['arppu']
                        second_last_arppu = prev_rows.iloc[-2]['arppu']
                        if not pd.isna(last_arppu) and not pd.isna(second_last_arppu) and second_last_arppu != 0:
                            df_exp.loc[idx, 'arppu_change'] = float((last_arppu - second_last_arppu) / second_last_arppu)
                        else:
                            df_exp.loc[idx, 'arppu_change'] = 0.0
    return df_exp


def get_experiment_features(baseline_features: list, experiment_type: str) -> list:
    features = list(baseline_features) if baseline_features else []
    if experiment_type in ["arppu", "arppu_change"] and 'last_arppu' not in features:
        features.append('last_arppu')
    if experiment_type == "arppu_change" and 'arppu_change' not in features:
        features.append('arppu_change')
    return features


def model_train_save_pipeline_rev(game, alpha, X_train, y_train):
    model = xgb.XGBRegressor(objective=quantile_loss(alpha), enable_categorical=True)
    numerical_columns = X_train.select_dtypes(include=['int64', 'int32', 'float64', 'float32']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[('num', StandardScaler(), numerical_columns)],
        remainder='passthrough'
    )
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    param_dist = {
        'model__n_estimators': randint(50, 300),
        'model__learning_rate': uniform(0.01, 0.09),
        'model__max_depth': randint(3, 8),
        'model__subsample': uniform(0.6, 0.4),
        'model__colsample_bytree': uniform(0.6, 0.4),
        'model__min_child_weight': randint(1, 6),
        'model__gamma': uniform(0, 0.2),
        'model__reg_alpha': uniform(0, 0.5),
        'model__reg_lambda': uniform(1, 1.5),
    }
    random_search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist,
        n_iter=20,
        scoring=make_scorer(mean_pinball_loss, alpha=alpha, greater_is_better=False),
        cv=TimeSeriesSplit(n_splits=3),
        n_jobs=-1,
        random_state=42,
    )
    random_search.fit(X_train, y_train)
    best_model = random_search.best_estimator_
    # Persist feature names before saving
    best_model.__dict__["feature_names_in_"] = X_train.columns
    with open(f'{game}_xgb_{alpha}_rev.pkl', 'wb') as f:
        cloudpickle.dump(best_model, f)
    y_train_pred = best_model.predict(X_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    return {"model": best_model, "rmse_train": rmse_train}


def augment_data(X, y, augmentation_factor, variation_pct):
    n_aug = int((augmentation_factor - 1) * len(X))
    if n_aug <= 0:
        return X, y
    # Match LTV/CAC script behavior: exclude only cohort/current month
    numeric_cols = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col]) and col not in ['cohort_month', 'current_month']]
    X_aug_list, y_aug_list = [], []
    for _ in range(n_aug):
        idx = np.random.randint(0, len(X))
        x_row = X.iloc[idx].copy()
        for col in numeric_cols:
            sign = np.random.choice([-1, 1])
            x_row[col] = x_row[col] * (1 + sign * (variation_pct / 100.0))
        X_aug_list.append(x_row)
        y_val = y.iloc[idx]
        sign = np.random.choice([-1, 1])
        y_aug_list.append(y_val * (1 + sign * (variation_pct / 100.0)))
    X_aug = pd.DataFrame(X_aug_list)
    y_aug = pd.Series(y_aug_list, index=X_aug.index)
    X_final = pd.concat([X, X_aug], ignore_index=True)
    y_final = pd.concat([y, y_aug], ignore_index=True)
    return X_final, y_final


def preprocess_m2(game):
    df = pd.read_csv(f'Latest Data/Segregated/{game}_Dataset_12_2024.csv')
    df['elapsed_months'].fillna(1, inplace=True)
    df = df[(df['total_cac'] != 0) & (df['total_cac'].notna())]
    df['elapsed_months'] -= 1
    # Special handling for G090 date formatting inconsistencies
    if game == "G090":
        if 'cohort' in df.columns:
            df['cohort'] = df['cohort'].astype(str).str.replace('-', '/')
        if 'transaction_period' in df.columns:
            df['transaction_period'] = df['transaction_period'].astype(str).str.replace('-', '/')
    df['total_cac'] = df.apply(
        lambda row: row['client_actual_spend'] if (pd.isnull(row['total_cac']) or row['total_cac'] == 0) and not pd.isnull(row['client_actual_spend']) else row['total_cac'],
        axis=1,
    )
    df['client_actual_spend'] = df.apply(
        lambda row: row['total_cac'] if (pd.isnull(row['client_actual_spend']) or row['client_actual_spend'] == 0) and not pd.isnull(row['total_cac']) else row['client_actual_spend'],
        axis=1,
    )
    df = df[~df['cohort'].str.endswith(('2021', '2022', '2020', '2019', '2018'))]
    df = df[~df['cohort'].str.startswith(('2021', '2022', '2020', '2019', '2018'))]
    df['cohort'] = pd.to_datetime(df['cohort'], format='%d/%m/%Y')
    df['transaction_period'] = pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce')
    df['cohort_month'] = df['cohort'].dt.month
    df['current_month'] = df.apply(calculate_current_month, axis=1)
    df['cohort_month'] = df['cohort_month'].astype('category')
    df['current_month'] = df['current_month'].astype('category')
    # Sort
    df = df.sort_values(by=['cohort', 'elapsed_months'])
    # LTV/CAC lags remain the drivers
    for lag in range(1, 4):
        df[f'ltv_lag_{lag}'] = df.groupby('cohort')['transaction_period_ltv_cac'].shift(lag)
    current_month_start = pd.Timestamp.today().normalize().replace(day=1)  # first day of current month as a Timestamp (fix C1)
    dft = df[(pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce') < current_month_start) & (df['elapsed_months'] > 0)].copy()
    return dft, df


def forecast_cumulative_net_revenue_variations_m2(game_id, best_model_expected, best_model_bad, best_model_good, df, forecast_months=12, epsilon=0.001):
    game_data = df[df['game_id'] == game_id].copy()
    if game_id == "G090":
        game_data = game_data[game_data['cohort'] >= pd.Timestamp("2024-12-01")]
    forecasted_results = []
    sp_ltv_cac = df[df['transaction_period_ltv_cac'] > 0]['transaction_period_ltv_cac'].min()
    if pd.isna(sp_ltv_cac) or not np.isfinite(sp_ltv_cac):
        sp_ltv_cac = epsilon

    for cohort in game_data['cohort'].unique():
        cohort_data = game_data[game_data['cohort'] == cohort].copy()
        m0_ltv_cac = cohort_data['m0_ltv_cac'].iloc[0]
        total_cac = cohort_data['client_actual_spend'].iloc[0]
        last_elapsed_month = int(cohort_data['elapsed_months'].max())

        # Elapsed 0 row (actual)
        elapsed_0_data = cohort_data.loc[cohort_data['elapsed_months'] == 0]
        if not elapsed_0_data.empty:
            transactional_actual_0 = elapsed_0_data['transaction_period_ltv_cac'].values[0]
            cumulative_actual_0 = m0_ltv_cac
            absolute_revenue_actual_0 = total_cac * transactional_actual_0
        else:
            transactional_actual_0 = 0
            cumulative_actual_0 = 0
            absolute_revenue_actual_0 = 0

        row_result_0 = {
            'game_id': game_id,
            'cohort': cohort,
            'elapsed_months': 0,
            'current_month': cohort.strftime('%Y-%m'),
            'transactional_actual': transactional_actual_0,
            'cumulative_actual': cumulative_actual_0,
            'absolute_revenue_actual': absolute_revenue_actual_0,
            'total_cac': total_cac,
            'transactional_bad': 0,
            'cumulative_bad': cumulative_actual_0,
            'absolute_revenue_bad': absolute_revenue_actual_0,
            'transactional_expected': 0,
            'cumulative_expected': cumulative_actual_0,
            'absolute_revenue_expected': absolute_revenue_actual_0,
            'transactional_good': 0,
            'cumulative_good': cumulative_actual_0,
            'absolute_revenue_good': absolute_revenue_actual_0,
        }
        forecasted_results.append(row_result_0)

        # Build actual cumulative history
        cumulative_actual = cumulative_actual_0
        actual_cumulative = []
        for month in range(1, last_elapsed_month + 1):
            historical_data = cohort_data.loc[cohort_data['elapsed_months'] == month, 'transaction_period_ltv_cac']
            transactional_actual = historical_data.values[0] if not historical_data.empty else 0
            cumulative_actual += transactional_actual
            actual_cumulative.append({
                'month': month,
                'transactional_actual': transactional_actual,
                'cumulative_actual': cumulative_actual,
                'absolute_revenue_actual': total_cac * transactional_actual,
                'total_cac': total_cac
            })

        # Forecasting with lags based on LTV/CAC values
        lag_1 = np.nan; lag_2 = np.nan; lag_3 = np.nan
        lag_1b = np.nan; lag_1g = np.nan; lag_2b = np.nan; lag_2g = np.nan; lag_3b = np.nan; lag_3g = np.nan
        cumulative_values = {key: cumulative_actual_0 for key in ['cumulative_bad', 'cumulative_expected', 'cumulative_good']}

        feature_names = best_model_expected.__dict__["feature_names_in_"]
        static_features = cohort_data.loc[cohort_data['elapsed_months'] == last_elapsed_month, feature_names].iloc[0].copy()
        for col in ['elapsed_months', 'current_month', 'ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3']:
            if col in static_features:
                static_features = static_features.drop(col)

        for i in range(1, last_elapsed_month + forecast_months + 1):
            elapsed_months = i
            features = static_features.copy()
            features['elapsed_months'] = elapsed_months
            cohort_start_date = pd.to_datetime(cohort_data['cohort'].iloc[0])
            current_date = cohort_start_date + pd.DateOffset(months=elapsed_months)
            features['current_month'] = current_date.month

            features1 = features.copy(); features2 = features.copy()
            features['ltv_lag_1'] = lag_1; features1['ltv_lag_1'] = lag_1b; features2['ltv_lag_1'] = lag_1g
            features['ltv_lag_2'] = lag_2; features1['ltv_lag_2'] = lag_2b; features2['ltv_lag_2'] = lag_2g
            features['ltv_lag_3'] = lag_3; features1['ltv_lag_3'] = lag_3b; features2['ltv_lag_3'] = lag_3g
            # ARPPU features handling
            if 'last_arppu' in feature_names:
                if i <= last_elapsed_month and 'arppu' in cohort_data.columns:
                    prev_month = i - 1
                    prev_series = cohort_data[cohort_data['elapsed_months'] == prev_month]['arppu']
                    last_arppu_val = float(prev_series.iloc[0]) if not prev_series.empty and not pd.isna(prev_series.iloc[0]) else 0.0
                else:
                    # future months: hold last known ARPPU at end of history
                    end_series = cohort_data[cohort_data['elapsed_months'] == last_elapsed_month]['arppu'] if 'arppu' in cohort_data.columns else pd.Series([0.0])
                    last_arppu_val = float(end_series.iloc[0]) if not end_series.empty and not pd.isna(end_series.iloc[0]) else 0.0
                features['last_arppu'] = last_arppu_val
                features1['last_arppu'] = last_arppu_val
                features2['last_arppu'] = last_arppu_val
            if 'arppu_change' in feature_names:
                if i <= last_elapsed_month and 'arppu' in cohort_data.columns:
                    prev1 = cohort_data[cohort_data['elapsed_months'] == (i - 1)]['arppu']
                    prev2 = cohort_data[cohort_data['elapsed_months'] == (i - 2)]['arppu']
                    if not prev1.empty and not prev2.empty and not pd.isna(prev1.iloc[0]) and not pd.isna(prev2.iloc[0]) and prev2.iloc[0] != 0:
                        arppu_change_val = float((prev1.iloc[0] - prev2.iloc[0]) / prev2.iloc[0])
                    else:
                        arppu_change_val = 0.0
                else:
                    arppu_change_val = 0.0
                features['arppu_change'] = arppu_change_val
                features1['arppu_change'] = arppu_change_val
                features2['arppu_change'] = arppu_change_val
            features_df = pd.DataFrame([features]); features_df1 = pd.DataFrame([features1]); features_df2 = pd.DataFrame([features2])

            if i <= last_elapsed_month:
                # Historical months: set predictions equal to actual LTV/CAC (like LTV post-processing)
                historical_data = cohort_data.loc[cohort_data['elapsed_months'] == i, 'transaction_period_ltv_cac']
                lag_value = float(historical_data.values[0]) if not historical_data.empty else epsilon
                lag_1 = lag_value; lag_1b = lag_value; lag_1g = lag_value
                predicted_val_bad = lag_value
                predicted_val_expected = lag_value
                predicted_val_good = lag_value
            else:
                # Model predicts revenue per 1000 spend; convert to LTV/CAC for lags
                pred_bad_raw = max(0, best_model_bad.predict(features_df1)[0])
                pred_expected_raw = max(0, best_model_expected.predict(features_df)[0])
                pred_good_raw = max(0, best_model_good.predict(features_df2)[0])
                predicted_val_bad = pred_bad_raw / 1000.0 if total_cac > 0 else 0
                predicted_val_expected = pred_expected_raw / 1000.0 if total_cac > 0 else 0
                predicted_val_good = pred_good_raw / 1000.0 if total_cac > 0 else 0

            x = [predicted_val_bad, predicted_val_expected, predicted_val_good]
            x.sort()
            predicted_val_bad, predicted_val_expected, predicted_val_good = x
            if predicted_val_good <= 0:
                predicted_val_good = sp_ltv_cac
            if predicted_val_expected <= 0:
                predicted_val_expected = predicted_val_good / 2
            if predicted_val_bad <= 0:
                predicted_val_bad = predicted_val_expected / 2

            # Update lags for next iteration
            lag_3 = lag_2; lag_2 = lag_1; lag_1 = predicted_val_expected
            lag_3b = lag_2b; lag_2b = lag_1b; lag_1b = predicted_val_bad
            lag_3g = lag_2g; lag_2g = lag_1g; lag_1g = predicted_val_good

            row_result = {
                'game_id': game_id,
                'cohort': cohort,
                'elapsed_months': elapsed_months,
                'current_month': current_date.strftime('%Y-%m'),
            }
            actual_data = next((ac for ac in actual_cumulative if ac['month'] == elapsed_months), None)
            row_result['transactional_actual'] = actual_data['transactional_actual'] if actual_data else None
            row_result['cumulative_actual'] = actual_data['cumulative_actual'] if actual_data else None
            row_result['absolute_revenue_actual'] = actual_data['absolute_revenue_actual'] if actual_data else None
            row_result['total_cac'] = actual_data['total_cac'] if actual_data else total_cac

            for model_type, predicted_val in zip(['bad', 'expected', 'good'], [predicted_val_bad, predicted_val_expected, predicted_val_good]):
                cumulative_key = f"cumulative_{model_type}"
                cum_val = row_result.get(cumulative_key, cumulative_values.get(cumulative_key, 0)) + predicted_val
                cumulative_values[cumulative_key] = cum_val
                row_result[f"transactional_{model_type}"] = predicted_val
                row_result[cumulative_key] = cum_val
                # Convert back to revenue using total_cac
                row_result[f"absolute_revenue_{model_type}"] = total_cac * predicted_val

            forecasted_results.append(row_result)

    forecast_df = pd.DataFrame(forecasted_results)
    return forecast_df


# --- Main Execution for M2 Net Revenue ---
if __name__ == "__main__":
    config_path = "Latest Data/Segregated/config2.json"
    try:
        with open(config_path, "r") as f:
            global_config = json.load(f)
    except Exception as e:
        print("No config found or error reading config2.json.")
        global_config = {}
    failed_games = []
    changed_features = {}
    for key, cfg in global_config.items():
        if key.startswith("M2_") and cfg.get("flag", 0) == 1:
            game_id = cfg["game_id"]
            selected_features = cfg["selected_features"]
            use_augmentation = cfg["use_augmentation"]
            augmentation_factor = cfg.get("augmentation_factor", 1.0)
            augmentation_pct = cfg.get("augmentation_pct", 0)
            print(f"Processing M2 Net Revenue configuration for {game_id} ...")
            try:
                dft, df_orig = preprocess_m2(game_id)
                missing_feats = [feat for feat in selected_features if feat not in dft.columns]
                if missing_feats:
                    print(f"Warning: For game {game_id}, the following features are missing and will be ignored: {missing_feats}")
                    changed_features[game_id] = missing_feats
                    selected_features = [feat for feat in selected_features if feat in dft.columns]
                # Iterate over ARPPU experiment types
                experiments = ["baseline", "arppu", "arppu_change"]
                for exp in experiments:
                    print(f"  ➤ Experiment: {exp}")
                    dft_exp = add_arppu_features(dft, exp)
                    feature_list = get_experiment_features(selected_features, exp)
                    train_cols = feature_list + ['ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3', 'elapsed_months']
                    X = dft_exp[train_cols]
                    # Target: revenue per 1000 spend
                    net_revenue = dft_exp['transaction_period_ltv_cac'] * dft_exp['total_cac']
                    y = np.where(dft_exp['total_cac'] > 0, net_revenue / (dft_exp['total_cac'] / 1000), 0)
                    y = pd.Series(y, index=dft_exp.index)
                    X_train = X; y_train = y
                    if use_augmentation and augmentation_factor > 1:
                        X_train, y_train = augment_data(X_train, y_train, augmentation_factor, augmentation_pct)

                    alphas = [0.25, 0.5, 0.75]
                    models_info = {}
                    for alpha in alphas:
                        print(f"    Training NET-REVENUE model (exp={exp}) with alpha={alpha} for game {game_id} ...")
                        result = model_train_save_pipeline_rev(game_id, alpha, X_train, y_train)
                        # Rename saved file to include experiment
                        src_name = f"{game_id}_xgb_{alpha}_rev.pkl"
                        dst_name = f"{game_id}_xgb_{alpha}_rev_{exp}.pkl"
                        try:
                            if os.path.exists(src_name):
                                os.replace(src_name, dst_name)
                        except Exception:
                            pass
                        models_info[alpha] = result

                    best_model_expected = models_info[0.5]["model"]
                    best_model_bad = models_info[0.75]["model"]
                    best_model_good = models_info[0.25]["model"]
                    forecast_df = forecast_cumulative_net_revenue_variations_m2(
                        game_id, best_model_expected, best_model_bad, best_model_good, df_orig, forecast_months=12
                    )
                    output_csv = f"{game_id}_predictions_rev_{exp}.csv"
                    forecast_df.to_csv(output_csv, index=False)
                    print(f"    Predictions (Net Revenue, {exp}) saved to {output_csv}")
            except Exception as e:
                print(f"Error processing game {game_id}: {e}")
                failed_games.append(game_id)
                continue
    if failed_games:
        print("\nThe following games failed during Net Revenue training:")
        for game in failed_games:
            print(game)
    if changed_features:
        print("\nThe following games had missing features for Net Revenue (ignored):")
        for game, features in changed_features.items():
            print(f"Game {game}: Missing features {features}")